In [1]:
import torch
import numpy as np

from dinosaw.utils import add_custom_font
from dinosaw.wrappers import get_model, get_models, ModelTypes, MODEL_NAMES, PretrainedViTWrapper
from dinosaw.linear_probe import do_linear_probe, RampTypes, LinearProbeResult, get_ramp, gen_sample_mask
# from dinosaw.models.vit_wrapper import PretrainedViTWrapper, MODEL_LIST
import dinosaw.utils as utils
from skimage.transform import resize

from os import listdir
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

from typing import Literal, TypeAlias

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = 'cuda:0'

/home/ronan/dino-saw/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
selected_models: tuple[ModelTypes, ...] = ('dinov2_s', 'dinov3_s+', 'alibi_coco_dinov2_s')
# selected_models: tuple[ModelTypes, ...] = ('dv_b', 'dv2_b', 'dv3_b', 'clip_b', 'vit_b_in', 'vit_b')

models = get_models(selected_models, DEVICE, True, '../../models/checkpoints',  conf_dir='../../models/dinov3')
S = models[selected_models[0]].stride

n_layers = 12
n_dims = models[selected_models[0]].embed_dim

2026-07-24 08:11:10 | I | factory.py                 : 152 | Building wrapper 'dinov2_s' on device cuda:0
2026-07-24 08:11:10 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='timm', model_arch='dinov2_s', pretrained=True, checkpoint_path=None, model_conf_path='../../models/dinov3', stride=None, remove_pos_embed=False, add_flash_attn=False, dynamic_img_size=True, dynamic_img_pad=False, modifications=[], dtype=torch.float32)
2026-07-24 08:11:10 | I | wrapper.py                 :  48 | Initialized PretrainedViTWrapper - name: '', arch: 'None', device: cuda:0, stride: (14, 14), patch_size: (14, 14), embed_dim: 384, num_blocks: 12
2026-07-24 08:11:10 | I | factory.py                 : 152 | Building wrapper 'dinov3_s+' on device cuda:0
2026-07-24 08:11:10 | I | factory.py                 : 131 | Building backbone with config: BackboneConfig(backbone_type='torch_hub', model_arch='dinov3_s+', pretrained=False, checkpoint_path='../../models/

In [3]:
def get_feature_list(
    model: PretrainedViTWrapper,
    pil_img: Image.Image,
    channel_last: bool = False,
    channel_blank: bool = False,
    N: int = 11
) -> list[np.ndarray]:
    # tr = utils.closest_resize(pil_img.height, pil_img.width, model.stride)
    # img_tensor = utils.convert_image(pil_img, tr, device_str=device, to_half=to_half)

    with torch.no_grad():
        embs = model.forward_intermediates(pil_img, list(range(N)))
    embs_np = [utils.to_numpy(emb.squeeze(0)) for emb in embs]
    if channel_blank:
        channels_to_blank = [47, 113, 117, 359]
        for emb_np in embs_np:
            emb_np[channels_to_blank, :, :] = 0
    if channel_last:
        embs_np = [np.transpose(emb_np, (1, 2, 0)) for emb_np in embs_np]

    return embs_np

In [4]:
ds_folder = 'data/linear_probe/homog_micros'
image_files = [f for f in listdir(ds_folder)]
n_imgs = len(image_files)

features: dict[ModelTypes, list[list[np.ndarray]]] = {key: [] for key in selected_models}

for key, model in models.items():
    for img_file in image_files:
        img_path = f'{ds_folder}/{img_file}'
        img = Image.open(img_path).convert('RGB')
        feat_list = get_feature_list(model, img, channel_last=True, N=n_layers)
        features[key].append(feat_list)

2026-07-24 08:11:11 | I | wrapper.py                 :  92 | Processing image, size: [699, 578]


2026-07-24 08:11:11 | I | wrapper.py                 : 151 | Forward Intermediates: x: [1,3,574,686] -> f: [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49] + [1,384,41,49]
2026-07-24 08:11:11 | I | wrapper.py                 :  92 | Processing image, size: [597, 596]
2026-07-24 08:11:11 | I | wrapper.py                 : 151 | Forward Intermediates: x: [1,3,588,588] -> f: [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42] + [1,384,42,42]
2026-07-24 08:11:11 | I | wrapper.py                 :  92 | Processing image, size: [800, 528]
2026-07-24 08:11:11 | I | wrapper.py                 : 151 | Forward Intermediates: x: [1,3,518,798] -> f: [1,384,37,57] + [1,384,37,57] + [1,384,37,57] + [1,384,37,57] + [1,384,37,57] + [1,384,37,57] + [1,384,37,5

In [5]:
AverageResult: TypeAlias = tuple[np.ndarray, np.ndarray, float, float, np.ndarray]
def average_results(results: list[LinearProbeResult]) -> AverageResult:
    channel_scores_arr = np.array([res['per_channel_scores'] for res in results])
    mean_channel_scores = np.mean(channel_scores_arr, axis=0)
    std_channel_scores = np.std(channel_scores_arr, axis=0)

    scores_arr = np.array([res['stack_r_squared'] for res in results])
    mean_score = np.mean(scores_arr, axis=0)
    std_score = np.std(scores_arr, axis=0)

    n_pred_dims = results[0]['stack_pred'].shape[-1]
    mean_pred = np.zeros((34, 34, n_pred_dims))

    for res in results:
        pred = resize(res['stack_pred'], mean_pred.shape, order=1)
        mean_pred += pred / len(results)

    return mean_channel_scores, std_channel_scores, mean_score, std_score, mean_pred

In [6]:
# ramps: tuple[RampTypes, ...] = ('lr', 'ud', 'diag', 'radial')
ramp: RampTypes = 'lr'
model_to_layer_results: dict[ModelTypes, list[AverageResult]] = {key: [] for key in selected_models}
MASK_CUTOFF_FRAC = 1
STEP = 6
RANDOM_MASK = True

for key, model in models.items():
    for layer in range(n_layers):
        image_results_for_layer = []
        for i in range(n_imgs):
            feats = features[key][i][layer]
            result = do_linear_probe(feats, ramp, probe_by_channel=True, mask_step=STEP, mask_cutoff_frac=MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
            image_results_for_layer.append(result)
        averaged = average_results(image_results_for_layer)
        model_to_layer_results[key].append(averaged)
        

In [7]:
model_to_layer_channel_scores: dict[ModelTypes, np.ndarray] = {key: None for key in selected_models}
for key in selected_models:
    layer_channel_scores = np.array([res[0] for res in model_to_layer_results[key]])
    model_to_layer_channel_scores[key] = layer_channel_scores

In [8]:
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection

def add_red_square_overlay(ax: plt.Axes, mask: np.ndarray, sf_h: float, sf_w: int) -> None:
    patches: list[Rectangle] = []
    y_inds, x_inds  = np.nonzero(mask)
    for x, y in zip(x_inds, y_inds):
        rect = Rectangle((x - sf_w / 2, y - sf_h / 2), sf_w, sf_h)
        patches.append(rect)
    pc = PatchCollection(patches, color='red', facecolor='none', lw=0.25)
    ax.add_collection(pc)

In [9]:
print(plt.rcParams["font.size"])
print(plt.rcParams["axes.labelsize"])
print(plt.rcParams["xtick.labelsize"])
print(plt.rcParams["ytick.labelsize"])

10.0
medium
medium
medium


In [10]:
n_rows, n_cols = 2, len(selected_models) +1
# FS = 15
# W, H = len(selected_models), 12

W, H = 2.25, 4  * 2.3
fig = plt.figure(figsize=(W, H))

plt.style.use("thesis.mplstyle")
plt.rcParams['text.usetex'] = False

add_custom_font('resources/fonts', 'Grotesk')

model_names: dict[ModelTypes, str] = {'dinov2_s': 'DINOv2', 'dinov3_s+': 'DINOv3', 'alibi_coco_dinov2_s': 'ALiBi-Dv2'}


# 'dinov2_s', 'dinov3_s+', 'alibi_coco_dinov2_s'

colors: dict[ModelTypes, str] = {
    'dinov2_s': '#5762D5',
    'dinov3_s+': '#fcba03',
    'alibi_coco_dinov2_s': '#16ce37',
}


w_spacing = [1/3 for _ in range(len(selected_models))] #[1/3, 1/3, 1/3, 1/4]
w_spacing.append(1/4)
h_spacing = [0.12, 0.6, ]
fig = plt.figure(figsize=(sum([W  * w for w in w_spacing ]), sum([H * h for h in h_spacing])))
gs = GridSpec(n_rows, n_cols, figure=fig, width_ratios=w_spacing, height_ratios=h_spacing, hspace=0.2, wspace=0.5)

h, w = 34, 34
ramp_arr = get_ramp(ramp, h, w)

ramp_ax = fig.add_subplot(gs[0, :-1])
ramp_ax.imshow(ramp_arr, aspect='equal', vmin=0, vmax=1, interpolation='nearest', cmap='viridis',)
ramp_ax.set_xticks([])
ramp_ax.set_yticks([])
ramp_ax.set_title('Target ramp', )

mask = gen_sample_mask((h, w), ramp, STEP, MASK_CUTOFF_FRAC, random_mask=RANDOM_MASK)
add_red_square_overlay(ramp_ax, mask, 1, 1)

# first_ax = fig.add_subplot(gs[1, 0])
fingerprint_axes = []
for i, key in enumerate(selected_models):
    ax = fig.add_subplot(gs[1, i])
    fingerprint_axes.append(ax)
    ax.imshow(model_to_layer_channel_scores[key].T, aspect='auto', vmin=0, vmax=0.5, cmap='viridis',)
    name = model_names[key]
    name = name.replace('(COCO)', '')
    weight = 700 if 'alibi' in key else 500
    ax.set_title(name,  weight=weight)
    ax.tick_params(labelsize=6)
    print(i, key)

    if i > 0:
        ax.set_yticks([])
    else:
        ax.set_ylabel('Channel', )

fingerprint_axes[1].set_xlabel('Layer', )
cbar_ax = fig.add_subplot(gs[1, -1])
cbar = fig.colorbar(ax.images[0], ax=cbar_ax, fraction=1, pad=0.08,)
cbar.set_label(r'Per-channel $R^2$ scores',  rotation=270, labelpad=20)
cbar_ax.set_axis_off()
cbar.ax.tick_params(labelsize=6)


SAVE = True
if SAVE:
    plt.savefig("saved/03.pdf", dpi=300, bbox_inches='tight')
    plt.close()

findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight normal, now using 300.
findfont: Failed to find font weight 500, now using 300.


findfont: Failed to find font weight normal, now using 300.


0 dinov2_s
1 dinov3_s+
2 alibi_coco_dinov2_s


<Figure size 225x920 with 0 Axes>